# IA Générative — TP 1 : Generative Adversarial Networks (GANs)
**FST — IDS/IGL 5 — 2026-2027**

- **Exercice 1** : GAN simple en **PyTorch** qui apprend une courbe sinusoïdale.
- **Exercice 2** : GAN dense en **TensorFlow/Keras** sur **MNIST** puis **Fashion-MNIST** (datasets téléchargés automatiquement par TensorFlow).

> Fonctionne en local (CPU suffit) ou sur Google Colab.

In [ ]:
# Décommenter si besoin
# !pip install torch tensorflow matplotlib numpy

# Exercice 1 — GAN sur une courbe sinusoïdale (PyTorch)

## 1. Préparation des données

In [ ]:
import torch
from torch import nn

import math
import matplotlib.pyplot as plt

In [ ]:
torch.manual_seed(111)

### Préparation des données d'apprentissage
Chaque exemple est un couple $(x_1, x_2)$ avec $x_1 \in [0, 2\pi]$ et $x_2 = \sin(x_1)$.

In [ ]:
train_data_length = 1024
train_data = torch.zeros((train_data_length, 2))
train_data[:, 0] = 2 * math.pi * torch.rand(train_data_length)
train_data[:, 1] = torch.sin(train_data[:, 0])
train_labels = torch.zeros(train_data_length)
train_set = [
    (train_data[i], train_labels[i]) for i in range(train_data_length)
]

In [ ]:
plt.plot(train_data[:, 0], train_data[:, 1], ".")
plt.title("Données réelles : y = sin(x)")
plt.show()

## 2. Chargement des données avec DataLoader

In [ ]:
batch_size = 32
train_loader = torch.utils.data.DataLoader(
    train_set, batch_size=batch_size, shuffle=True
)

## 3. Implémentation du discriminateur
MLP 2 → 256 → 128 → 64 → 1 avec ReLU et Dropout(0.3). La sortie `Sigmoid` donne la probabilité que le point soit **réel**.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(2, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        output = self.model(x)
        return output

In [ ]:
discriminator = Discriminator()

## 4. Implémentation du générateur
Il prend un bruit gaussien de dimension 2 (espace latent) et produit un point $(x_1, x_2)$. Pas d'activation en sortie : les coordonnées peuvent prendre n'importe quelle valeur.

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 2),
        )

    def forward(self, x):
        output = self.model(x)
        return output

generator = Generator()

## 5. Apprentissage du modèle

In [ ]:
lr = 0.001
num_epochs = 300
loss_function = nn.BCELoss()

In [ ]:
optimizer_discriminator = torch.optim.Adam(discriminator.parameters(), lr=lr)
optimizer_generator = torch.optim.Adam(generator.parameters(), lr=lr)

In [ ]:
hist_d, hist_g = [], []

for epoch in range(num_epochs):
    for n, (real_samples, _) in enumerate(train_loader):
        # Data for training the discriminator
        real_samples_labels = torch.ones((batch_size, 1))
        latent_space_samples = torch.randn((batch_size, 2))
        generated_samples = generator(latent_space_samples)
        generated_samples_labels = torch.zeros((batch_size, 1))
        all_samples = torch.cat((real_samples, generated_samples))
        all_samples_labels = torch.cat(
            (real_samples_labels, generated_samples_labels)
        )

        # Training the discriminator
        discriminator.zero_grad()
        output_discriminator = discriminator(all_samples)
        loss_discriminator = loss_function(
            output_discriminator, all_samples_labels
        )
        loss_discriminator.backward()
        optimizer_discriminator.step()

        # Data for training the generator
        latent_space_samples = torch.randn((batch_size, 2))

        # Training the generator
        generator.zero_grad()
        generated_samples = generator(latent_space_samples)
        output_discriminator_generated = discriminator(generated_samples)
        loss_generator = loss_function(
            output_discriminator_generated, real_samples_labels
        )
        loss_generator.backward()
        optimizer_generator.step()

        # Show loss
        if epoch % 10 == 0 and n == batch_size - 1:
            print(f"Epoch: {epoch} Loss D.: {loss_discriminator}")
            print(f"Epoch: {epoch} Loss G.: {loss_generator}")

    hist_d.append(loss_discriminator.item())
    hist_g.append(loss_generator.item())

> **Remarque :** `n == batch_size - 1` fonctionne car il y a exactement 1024 / 32 = 32 lots par époque : on affiche donc la perte du dernier lot.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(hist_d, label="Discriminateur")
plt.plot(hist_g, label="Générateur")
plt.xlabel("Époque"); plt.ylabel("Perte BCE"); plt.legend()
plt.title("Courbes d'apprentissage")
plt.show()

Les deux pertes se stabilisent autour de ~0,69 (= ln 2) : le discriminateur ne distingue plus les vrais points des faux (il répond ≈ 0,5), signe d'un équilibre.

## 6. Vérification de l'échantillon généré

In [ ]:
latent_space_samples = torch.randn(100, 2)
generated_samples = generator(latent_space_samples)

In [ ]:
generated_samples = generated_samples.detach()
plt.plot(train_data[:, 0], train_data[:, 1], ".", alpha=0.15, label="Réel")
plt.plot(generated_samples[:, 0], generated_samples[:, 1], ".", label="Généré")
plt.legend()
plt.title("Échantillons générés vs réels")
plt.show()

# Exercice 2 — GAN sur MNIST puis Fashion-MNIST (TensorFlow / Keras)

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Évite que TensorFlow réserve toute la mémoire GPU (utile sur Colab)
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

tf.random.set_seed(42)
np.random.seed(42)
print("TensorFlow", tf.__version__, "| GPU :", tf.config.list_physical_devices("GPU"))

## 1. Chargement et préparation de MNIST
Images 28×28 en niveaux de gris, normalisées dans **[-1, 1]** pour correspondre à la sortie `tanh` du générateur.

In [ ]:
def charger_dataset(nom="mnist"):
    if nom == "mnist":
        (x_train, _), _ = keras.datasets.mnist.load_data()
    else:
        (x_train, _), _ = keras.datasets.fashion_mnist.load_data()
    x_train = x_train.astype("float32")
    x_train = (x_train - 127.5) / 127.5          # [0, 255] -> [-1, 1]
    x_train = np.expand_dims(x_train, axis=-1)   # (N, 28, 28, 1)
    return x_train

x_train = charger_dataset("mnist")
print(x_train.shape, x_train.min(), x_train.max())

plt.figure(figsize=(10, 1.5))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(x_train[i, :, :, 0], cmap="gray"); plt.axis("off")
plt.suptitle("Exemples réels MNIST")
plt.show()

## 2. Générateur
Architecture conforme au résumé de l'énoncé :

| Couche | Sortie | Paramètres |
|---|---|---|
| Dense (LeakyReLU 0.2) | 128 | 100×128 + 128 = 12 928 |
| BatchNormalization | 128 | 512 |
| Dense (LeakyReLU 0.2) | 256 | 33 024 |
| BatchNormalization | 256 | 1 024 |
| Dense (tanh) | 784 | 201 488 |
| Reshape | 28×28×1 | 0 |

Les activations sont passées **dans** les couches `Dense` (c'est pour cela qu'elles n'apparaissent pas comme couches séparées dans le résumé).

In [ ]:
z_dim = 100
img_shape = (28, 28, 1)

def build_generator(z_dim=z_dim):
    model = keras.Sequential(name="generator")
    model.add(keras.Input(shape=(z_dim,)))
    model.add(layers.Dense(128, activation=layers.LeakyReLU(0.2)))
    model.add(layers.BatchNormalization(momentum=0.8))
    model.add(layers.Dense(256, activation=layers.LeakyReLU(0.2)))
    model.add(layers.BatchNormalization(momentum=0.8))
    model.add(layers.Dense(28 * 28 * 1, activation="tanh"))
    model.add(layers.Reshape(img_shape))
    return model

build_generator().summary()

## 3. Discriminateur
| Couche | Sortie | Paramètres |
|---|---|---|
| Flatten | 784 | 0 |
| Dense (LeakyReLU 0.2) | 128 | 784×128 + 128 = 100 480 |
| Dropout(0.3) | 128 | 0 |
| Dense (LeakyReLU 0.2) | 64 | 8 256 |
| Dropout(0.3) | 64 | 0 |
| Dense (sigmoid) | 1 | 65 |

La sortie est la probabilité que l'image soit **réelle**.

In [ ]:
def build_discriminator(img_shape=img_shape):
    model = keras.Sequential(name="discriminator")
    model.add(keras.Input(shape=img_shape))
    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation=layers.LeakyReLU(0.2)))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(64, activation=layers.LeakyReLU(0.2)))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(1, activation="sigmoid"))
    return model

build_discriminator().summary()

## 4. Définition et compilation des modèles
- `z_dim = 100`, optimiseur **Adam**, `learning_rate = 0.0002`, perte **binary_crossentropy**.
- `beta_1 = 0.5` : valeur recommandée pour les GANs (comme dans le notebook du cours).
- Le **GAN combiné** = générateur → discriminateur. Il sert uniquement à entraîner le générateur, donc le discriminateur y est **gelé**.

> ⚠️ **Keras 3 (TensorFlow ≥ 2.16)** : `trainable` n'est plus « figé » au moment du `compile`. Si on met `discriminator.trainable = False` une seule fois, le discriminateur **n'apprend plus du tout**, même avec son propre `train_on_batch`.
> On bascule donc `trainable` explicitement avant chaque étape (fonctionne aussi avec Keras 2).

In [ ]:
def build_models(z_dim=z_dim, lr=0.0002):
    discriminator = build_discriminator()
    discriminator.compile(optimizer=keras.optimizers.Adam(learning_rate=lr, beta_1=0.5),
                          loss="binary_crossentropy", metrics=["accuracy"])

    generator = build_generator(z_dim)

    discriminator.trainable = False           # gelé dans le modèle combiné
    z = keras.Input(shape=(z_dim,))
    gan = keras.Model(z, discriminator(generator(z)), name="gan")
    gan.compile(optimizer=keras.optimizers.Adam(learning_rate=lr, beta_1=0.5),
                loss="binary_crossentropy")
    return generator, discriminator, gan

generator, discriminator, gan = build_models()
gan.summary()

## 5. Entraînement du GAN
Pour chaque lot :
1. **Discriminateur** : un demi-lot d'images réelles (étiquette 1) + un demi-lot d'images générées (étiquette 0).
2. **Générateur** (via le GAN combiné) : du bruit avec l'étiquette **1** → le générateur apprend à tromper le discriminateur.

In [ ]:
def sample_images(generator, n=10, titre="", seed=None):
    rng = np.random.default_rng(seed)
    z = rng.normal(0, 1, (n, z_dim)).astype("float32")
    imgs = generator.predict(z, verbose=0)
    imgs = 0.5 * imgs + 0.5                     # [-1, 1] -> [0, 1]
    plt.figure(figsize=(n, 1.3))
    for i in range(n):
        plt.subplot(1, n, i + 1)
        plt.imshow(imgs[i, :, :, 0], cmap="gray"); plt.axis("off")
    plt.suptitle(titre)
    plt.show()


def train_gan(generator, discriminator, gan, x_train, epochs=30, batch_size=128, sample_every=5):
    half = batch_size // 2
    n_batches = x_train.shape[0] // batch_size
    history = {"d_loss": [], "d_acc": [], "g_loss": []}

    for epoch in range(1, epochs + 1):
        idx_all = np.random.permutation(x_train.shape[0])
        d_l, d_a, g_l = [], [], []
        for b in range(n_batches):
            # ---- 1) Discriminateur
            real = x_train[idx_all[b * half:(b + 1) * half]]
            z = np.random.normal(0, 1, (half, z_dim)).astype("float32")
            fake = generator.predict(z, verbose=0)
            X = np.concatenate([real, fake])
            y = np.concatenate([np.ones((half, 1)), np.zeros((half, 1))])

            discriminator.trainable = True
            d_loss, d_acc = discriminator.train_on_batch(X, y)[:2]

            # ---- 2) Générateur (discriminateur gelé)
            discriminator.trainable = False
            z = np.random.normal(0, 1, (batch_size, z_dim)).astype("float32")
            g_loss = gan.train_on_batch(z, np.ones((batch_size, 1)))
            g_loss = g_loss[0] if isinstance(g_loss, (list, tuple)) else g_loss

            d_l.append(d_loss); d_a.append(d_acc); g_l.append(float(g_loss))

        history["d_loss"].append(np.mean(d_l))
        history["d_acc"].append(np.mean(d_a))
        history["g_loss"].append(np.mean(g_l))
        print(f"Époque {epoch:3d}/{epochs} | D loss {history['d_loss'][-1]:.4f} "
              f"| D acc {100 * history['d_acc'][-1]:5.1f}% | G loss {history['g_loss'][-1]:.4f}")

        if epoch % sample_every == 0 or epoch == 1:
            sample_images(generator, titre=f"Époque {epoch}", seed=0)
    return history


def plot_history(history, titre=""):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(history["d_loss"], label="Discriminateur")
    ax[0].plot(history["g_loss"], label="Générateur")
    ax[0].set_title("Pertes"); ax[0].set_xlabel("Époque"); ax[0].legend()
    ax[1].plot(history["d_acc"], color="tab:green")
    ax[1].set_title("Exactitude du discriminateur"); ax[1].set_xlabel("Époque")
    fig.suptitle(titre)
    plt.show()

In [ ]:
EPOCHS = 30        # augmenter (ex. 50-100) pour de meilleurs résultats
BATCH_SIZE = 128

history_mnist = train_gan(generator, discriminator, gan, x_train,
                          epochs=EPOCHS, batch_size=BATCH_SIZE, sample_every=5)
plot_history(history_mnist, "MNIST")

## 6. Génération et visualisation

In [ ]:
def show_grid(generator, rows=4, cols=10, titre=""):
    z = np.random.normal(0, 1, (rows * cols, z_dim)).astype("float32")
    imgs = 0.5 * generator.predict(z, verbose=0) + 0.5
    plt.figure(figsize=(cols, rows))
    for i in range(rows * cols):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(imgs[i, :, :, 0], cmap="gray"); plt.axis("off")
    plt.suptitle(titre)
    plt.show()

show_grid(generator, titre="Chiffres générés — MNIST")

## 7. Mêmes étapes sur Fashion-MNIST
On réutilise exactement les mêmes fonctions : seul le dataset change. De nouveaux modèles sont créés pour repartir de zéro.

In [ ]:
x_train_fashion = charger_dataset("fashion")

plt.figure(figsize=(10, 1.5))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(x_train_fashion[i, :, :, 0], cmap="gray"); plt.axis("off")
plt.suptitle("Exemples réels Fashion-MNIST")
plt.show()

In [ ]:
generator_f, discriminator_f, gan_f = build_models()

history_fashion = train_gan(generator_f, discriminator_f, gan_f, x_train_fashion,
                            epochs=EPOCHS, batch_size=BATCH_SIZE, sample_every=5)
plot_history(history_fashion, "Fashion-MNIST")

In [ ]:
show_grid(generator_f, titre="Vêtements générés — Fashion-MNIST")

**Observation :** avec un générateur uniquement dense (pas de convolutions), les chiffres MNIST deviennent reconnaissables en quelques époques, alors que les vêtements de Fashion-MNIST restent plus bruités (textures et formes plus complexes).
Un **DCGAN** (couches convolutionnelles, TP 2) améliore nettement la qualité.